# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashelesc/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections `in order` — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- One row = one `content_id`.
- No date column, the window is implied by the `_90d`, `_last_30d`, `_prev_30d` in the column names.
- The `_last_30d` and `_prev_30d` together account for the most recent 60 days inside each 90-day total.

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

In [2]:
assert len(df) == df.content_id.nunique()
print(f"{len(df)} rows, {df.content_id.nunique()} unique content_id")

days_coverage = (df.impressions_last_30d + df.impressions_prev_30d) / df.impressions_90d
days_coverage.describe()

30000 rows, 30000 unique content_id


count    30000.000000
mean         0.562472
std          0.245485
min          0.000000
25%          0.416667
50%          0.570195
75%          0.709814
max          1.000000
dtype: float64

## 2. Fields: feature / label / context / excluded

- Label: `is_declining_label`
- Features:
    - `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`,
    - `days_with_impressions`, `days_with_sessions`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`,
    - `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `search_volume`, `competition`, `cpc`, `content_type`, `main_intent`,
    - `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `scroll_rate`, `ctr`, `avg_position`, `engagement_rate`, `ai_traffic_pct`.
- Context: kept in the dataframe for grouping/splitting/reporting, not fed to the model.
    - `client_id` — the train/test split key (split by client to avoid leakage), not a predictive signal.
    - `content_id` — grain identifier, not a feature.
    - `provider_used`, `model_used` — describe how the content was produced, not a search-performance signal.
    - `competition_level` — binned version of `competition`.
    - `age_tier`, `age_tier_order`— binned version of `content_age_days`.
    - `word_count_tier` — binned version of `word_count`.
    - `char_count_tier` — binned version of `char_count`.
    - `freshness_tier` — binned version of `days_since_last_update`.
    - `position_tier` — binned version of `avg_position`.
    - `impression_tier` — binned version of `impressions_90d`.
- Excluded:
    - `trend_pct` — causes leakage, a shallow tree trained in `01_first_look_and_discovery.ipynb` with this column reached `Precision@50 = 1.000` which indicates overfitting.
    - `trend_direction` — defines the label, `is_declining_label` is derived from this column.

In [3]:
label_column = 'is_declining_label'
feature_columns = [
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'search_volume', 'competition', 'cpc', 'content_type', 'main_intent',
    'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'scroll_rate', 'ctr', 'avg_position', 'engagement_rate', 'ai_traffic_pct'
]
context_columns = [
    'client_id', 'content_id', 'provider_used', 'model_used', 'age_tier', 'age_tier_order', 'freshness_tier',
    'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'competition_level'
]
excluded_columns = ['trend_pct', 'trend_position']

# the dataset has 45 features, 44 without the label column.
assert len(feature_columns) + len(context_columns) + len(excluded_columns) == 44

## 3. Verify it with queries (grain, counts, missing values, windows)

- `Grain` (Section 1) — one row per `content_id`.
- `Window overlap` (Section 1) — `_last_30d` + `_prev_30d`
   cover the most recent 60 days unevenly; the oldest third of the
   90-day window isn't broken out separately.
- `Full 90-day history for every row` (Section 1) — rules out
   "unbalanced history" as a limit.
- `Nulls on planned features` — check before training.
- `Raw/tier pairing` — confirm every `_tier`/`_level` column named
   in Section 2 as context really is a binned view of its raw
   counterpart, not an independent signal.

In [4]:
# Grain
assert len(df) == df.content_id.nunique()
print(f"Grain OK: {len(df)} rows, {df.content_id.nunique()} unique content_id")
print("")

# Window coverage
days_coverage = (df.impressions_last_30d + df.impressions_prev_30d) / df.impressions_90d
print(days_coverage.describe())
print("")

# Full 90-day history check
print(df.content_age_days.describe())
print("")

# Nulls on planned feature columns
print(df[feature_columns].isnull().sum().sort_values(ascending = False).head(10))
print("")

# Raw/tier pairing spot-check (example: age)
print(df.groupby("age_tier")["content_age_days"].agg(["min", "max", "count"]))

Grain OK: 30000 rows, 30000 unique content_id

count    30000.000000
mean         0.562472
std          0.245485
min          0.000000
25%          0.416667
50%          0.570195
75%          0.709814
max          1.000000
dtype: float64

count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64

char_count         7699
word_count         7699
competition        2468
cpc                2468
search_volume      2468
main_intent        2374
scroll_rate         125
ai_sessions_90d       0
impressions_90d       0
clicks_90d            0
dtype: int64

          min  max  count
age_tier                 
181-365   181  362  11368
31-90      90   90    492
365+      372  564   6360
91-180     91  180  11780


## 4. Data limits

- `Unbalanced history`: none of the pages has a `content_age_days` that's less than 90 days old, this indicates that there's no unbalanced history in this dataset.
- `GSC-only rows`: `len(gsc_only) = 0` indicates that none of the pages have search impressions, but 0 recorded sessions.
- `Window overlaps`: `days_coverage` has a mean of `0.562472`, which indicates that `impressions_last_30d` and `impressions_prev_30d` cover only about `56.25%` of each page's `_90d`. This means that the oldest third of the window is disproportionately represented, and not separately inspectable.

- `What this means`: since every row already has a 90-day history bar, this dataset cannot tell you how new pages behave. Any model trained on this dataset should not be trusted on pages younger than 90 days.

In [5]:
# Unbalanced history
print(df.content_age_days.describe())
print(f"{(df.content_age_days < 90).sum()} rows younger than the 90-day window itself")
print()

# GSC-only early rows
gsc_only = df[(df.impressions_90d > 0) & (df.sessions_90d.fillna(0) == 0)]
print(f"{len(gsc_only)} rows with impressions, but 0 recorded sessions")

# Window overlaps
days_coverage.describe()

count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64
0 rows younger than the 90-day window itself

0 rows with impressions, but 0 recorded sessions


count    30000.000000
mean         0.562472
std          0.245485
min          0.000000
25%          0.416667
50%          0.570195
75%          0.709814
max          1.000000
dtype: float64

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.